In [1]:
import os
import numpy as np; np.set_printoptions(linewidth=150)
import torch; torch.set_printoptions(linewidth=150)
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import pickle

In [2]:
from environment import CliffWalk, InfiniteWalk
from episodes import Episode, collect_episodes, EpisodeCollection
from belief_decoders import NonLinBeliefDecoder, LinBeliefDecoder, decode_training, decode_visualisation, estimate_entropy
from belief_rnn import BeliefRNN, RewardReadout, ValueReadout, NextLatentPredictor, ObsReadout
from train import train, plot_validate

In [4]:
N, M = 3, 8
cliff = CliffWalk(n=N, m=M, self_transition_prob=0.1, gamma=1.0)
policy = cliff.get_optimal_policy(epsilon=0.3)

# Collect episodes from the environment using the target policy
episode_list = collect_episodes(cliff, policy, num_episodes=500000)


episodes = EpisodeCollection(episode_list)
print(np.mean(episodes.ep_lengths))

11.86017


In [5]:
history_vtable = episodes.get_history_values(gamma=1.0)



History value estimation complete.
881321 unique histories found in dataset.


In [6]:
with open("history_value_st01_ep03.pkl", "wb") as f:
    pickle.dump(history_vtable, f)

In [7]:
print("Empirical State Value function:")
value_empirical = episodes.get_monte_carlo_state_values(gamma=cliff.gamma)
print(np.flip(value_empirical.reshape((cliff.n, cliff.m)), axis=0))

Empirical State Value function:
[[  1.86   2.1    2.55   3.02   3.53   4.32   6.13   9.35]
 [ -0.87  -1.18  -0.7   -0.13   0.46   1.08   2.91   9.5 ]
 [ -1.34 -10.   -10.   -10.   -10.   -10.   -10.    10.  ]]


In [8]:
with open("state_value_st01_ep03.pkl", "wb") as g:
    pickle.dump(value_empirical, g)